# 00_setup
Creates the Unity Catalog objects for the pipeline (one schema per medallion layer) and tests whether this workspace can reach the Wikimedia APIs.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS nba_demand COMMENT 'NBA Europe demand monitoring pipeline';
CREATE SCHEMA IF NOT EXISTS nba_demand.bronze COMMENT 'Raw Wikimedia API responses, stored as received. Never cleaned.';
CREATE SCHEMA IF NOT EXISTS nba_demand.silver COMMENT 'Cleaned daily and weekly pageview series.';
CREATE SCHEMA IF NOT EXISTS nba_demand.gold COMMENT 'Forecast band vs actual. Read by the dashboard and Genie.';
CREATE VOLUME IF NOT EXISTS nba_demand.bronze.landing COMMENT 'Raw JSON files, used only if the API must be called from outside Databricks.';

In [0]:
import requests

USER_AGENT = "nba-europe-demand-monitoring/0.1 (github.com/razvan-m-30; student project)"
tests = {
    "mediawiki (title lookup)": (
        "https://fr.wikipedia.org/w/api.php",
        {"action": "query", "titles": "Basket-ball|National Basketball Association",
         "redirects": 1, "format": "json"},
    ),
    "pageviews (1-7 Jan 2019)": (
        "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
        "fr.wikipedia/all-access/user/Basket-ball/daily/20190101/20190107",
        None,
    ),
}
for name, (url, params) in tests.items():
    try:
        r = requests.get(url, params=params, headers={"User-Agent": USER_AGENT}, timeout=20)
        print(f"{name}: HTTP {r.status_code}\n{r.text[:400]}\n")
    except Exception as e:
        print(f"{name}: FAILED {type(e).__name__}: {str(e)[:400]}\n")